# 서울 격자 단위 PM10 통합 시각화

**모델 구성**
- **Ambient PM10**: HiddenExtension V5-base (ST-GNN → 250m 격자)
- **Road PM10**: RoadExtension V3 TwoStage (시간 LightGBM + 공간 Ridge)
- **Combined PM10**: Ambient + road_struc% × Road

**셀 3에서 날짜·시간을 설정한 뒤 전체 실행**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from sklearn.impute import SimpleImputer
import joblib, os, sys, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/workspace/ST-GNN Modeling/RoadExtension_V3')
from preprocess import RoadPMTransformer
from config import TEMPORAL_STAGE_FEATS, SPATIAL_STAGE_FEATS

# ── 경로 설정 ─────────────────────────────────────────────────────────────
GRID_BASE   = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본'
GRID_CSV    = os.path.join(GRID_BASE, '격자_250m_4326.csv')
LUR_CSV     = os.path.join(GRID_BASE, '격자_250m_4326_with_lur.csv')
BLDG_NPY    = os.path.join(GRID_BASE, 'building_stats_static.npy')
TS_ALL      = os.path.join(GRID_BASE, 'timestamps_all.npy')

V5_TRAIN    = '/workspace/ST-GNN Modeling/HiddenExtension_V5/checkpoints/V5-base/grid_pm10_train.npy'
V5_VAL      = '/workspace/ST-GNN Modeling/HiddenExtension_V5/checkpoints/V5-base/grid_pm10_val.npy'
V5_TEST     = '/workspace/ST-GNN Modeling/HiddenExtension_V5/checkpoints/V5-base/grid_pm10_test.npy'
TS_LOOKUP   = '/workspace/ST-GNN Modeling/RoadExtension_V2/checkpoints/v5_ts_lookup.csv'
MODEL_PATH  = '/workspace/ST-GNN Modeling/RoadExtension_V3/checkpoints/models/two_stage.pkl'
FEAT_TR_CSV = '/workspace/ST-GNN Modeling/RoadExtension_V2/checkpoints/features_train.csv'
S3_CSV      = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/ST-GNN/feature_scenarios/S3_transport_pm10_pollutants.csv'

print('경로 설정 완료')

In [ ]:
# ── 정적 데이터 로드 (1회만 실행) ─────────────────────────────────────────

# 격자 좌표 + LUR
grid_df = pd.read_csv(GRID_CSV)
lur_df  = pd.read_csv(LUR_CSV)
bldg    = np.load(BLDG_NPY)  # (G, 3): elev_mean, sum_area, sum_height

# LUR 컬럼 소문자 통일
lur_df.columns = lur_df.columns.str.lower()

# 2D 격자 인덱스 (row, col)
grid_df['col'] = ((grid_df['CELL_X'] - grid_df['CELL_X'].min()) // 250).astype(int)
grid_df['row'] = ((grid_df['CELL_Y'] - grid_df['CELL_Y'].min()) // 250).astype(int)
NR = int(grid_df['row'].max()) + 1
NC = int(grid_df['col'].max()) + 1

# 전체 격자 공간 피처 행렬 구성 (G=10125)
G = len(grid_df)
spatial_feat_names = ['buildings','greenspace','road_struc','river_zone',
                      'ndvi','ibi','elev_mean','sum_area','sum_height']
X_spatial_base = np.column_stack([
    lur_df['buildings'].values,
    lur_df['greenspace'].values,
    lur_df['road_struc'].values,
    lur_df['river_zone'].values,
    lur_df['ndvi'].values,
    lur_df['ibi'].values,
    bldg[:, 0],  # elev_mean
    bldg[:, 1],  # sum_area
    bldg[:, 2],  # sum_height
])  # (G, 9)

# V5 ambient PM10 로드
v5_data = {
    'train': np.load(V5_TRAIN),
    'val':   np.load(V5_VAL),
    'test':  np.load(V5_TEST),
}
ts_lookup = pd.read_csv(TS_LOOKUP)
ts_lookup['dt'] = pd.to_datetime(ts_lookup['timestamp'])

# TwoStage 모델 로드
transformer, m1_temporal, imp_spatial, m2_spatial = joblib.load(MODEL_PATH)

# 월별 기온/습도 평균 (Stage1 격자 추론용)
feat_tr = pd.read_csv(FEAT_TR_CSV, parse_dates=['date'])
feat_tr['month'] = feat_tr['date'].dt.month
weather_monthly = feat_tr.groupby('month')[['기온','습도']].mean()

# 관측소 위치 (GNN 40개 노드)
s3_df = pd.read_csv(S3_CSV, nrows=5000)
stations = s3_df.groupby('측정소명')[['위도','경도']].first().reset_index()

print('데이터 로드 완료')
print('  격자 수: {:,}'.format(G))
print('  2D 크기: {} × {}'.format(NR, NC))
print('  V5 test 기간: {} ~ {}'.format(
    ts_lookup[ts_lookup['split']=='test']['dt'].min(),
    ts_lookup[ts_lookup['split']=='test']['dt'].max()))
print('  관측소 수: {}'.format(len(stations)))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ★ 여기서 날짜·시간 설정 ★
# V5 test 기간: 2025-07-10 ~ 2025-10-31 (시간별)
# V5 val  기간: 2025-03-17 ~ 2025-07-09
# V5 train 기간: 2023-10-01 ~ 2025-03-17
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TARGET_DT = '2025-10-01 10:00:00'

# 시각화 옵션
SAVE_PATH   = '/workspace/ST-GNN Modeling/HiddenExtension_V5/seoul_pm10_map.png'
SHOW_STATIONS = True   # 관측소 위치 표시
FIG_DPI     = 200

print('대상 시간:', TARGET_DT)

In [ ]:
# ── 추론: 선택 시간에 대해 전체 격자 PM10 계산 ───────────────────────────

target_dt = pd.Timestamp(TARGET_DT)

# 1. V5 Ambient PM10 획득
match = ts_lookup[ts_lookup['dt'] == target_dt]
if len(match) == 0:
    # 가장 가까운 시간 찾기
    diff = (ts_lookup['dt'] - target_dt).abs()
    match = ts_lookup.loc[[diff.idxmin()]]
    print('주의: 정확한 시간 없음 → 가장 가까운 시간 사용:', match['dt'].values[0])

row_info = match.iloc[0]
split     = row_info['split']
local_idx = int(row_info['local_idx'])
ambient_pm10_all = v5_data[split][local_idx]  # (G,)

actual_dt = row_info['dt']
month  = actual_dt.month
hour   = actual_dt.hour
weekday = actual_dt.weekday()
month_rad = 2 * np.pi * month / 12
hour_rad  = 2 * np.pi * hour  / 24
season_map = {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
season = season_map[month]

# 기온/습도: 월별 평균 사용
temp   = float(weather_monthly.loc[month, '기온'])
humid  = float(weather_monthly.loc[month, '습도'])
is_dry = int(humid < 40)

print('추론 시간: {}  (split={}, local_idx={})'.format(actual_dt, split, local_idx))
print('  month={}, hour={}, weekday={}, season={}'.format(month, hour, weekday, season))
print('  기온={:.1f}°C, 습도={:.1f}%, is_dry={}'.format(temp, humid, is_dry))
print('  ambient PM10: {:.2f} ~ {:.2f}  mean={:.2f}'.format(
    ambient_pm10_all.min(), ambient_pm10_all.max(), ambient_pm10_all.mean()))

# 2. Stage1 — 시간 피처로 모든 격자에 동일한 시간 기준값 계산
temporal_feats_order = ['month_sin','month_cos','hour_sin','hour_cos',
                         'weekday','is_weekend','season','기온','습도','is_dry']
X_temp = np.array([[
    np.sin(month_rad), np.cos(month_rad),
    np.sin(hour_rad),  np.cos(hour_rad),
    weekday, int(weekday >= 5), season,
    temp, humid, is_dry
]] * G, dtype=np.float32)  # (G, 10) — 모든 격자 동일

stage1_bc = m1_temporal.predict(X_temp)  # (G,)

# 3. Stage2 — 공간 피처 (LUR + ambient_pm + traffic=NaN)
traffic_col = np.full(G, np.nan)  # 격자별 교통량 없으면 NaN (Ridge가 median으로 대체)
X_spat = np.column_stack([
    X_spatial_base,          # 9개: LUR
    ambient_pm10_all,        # ambient_pm10
    traffic_col,             # traffic
])  # (G, 11)

X_spat_imp = imp_spatial.transform(X_spat)   # NaN → median
stage2_resid = m2_spatial.predict(X_spat_imp)  # (G,)

# 4. 최종 Road PM10
final_bc    = stage1_bc + stage2_resid
road_pm_all = np.clip(transformer.inverse_transform(final_bc), 0, None)  # (G,)

# 5. Combined PM10 = ambient + road_struc_fraction × road_pm
road_struc_frac = lur_df['road_struc'].values / 100.0  # 0~1 스케일
combined_pm_all = ambient_pm10_all + road_struc_frac * road_pm_all

print('\nRoad PM10: {:.2f} ~ {:.2f}  mean={:.2f}'.format(
    road_pm_all.min(), road_pm_all.max(), road_pm_all.mean()))
print('Combined:  {:.2f} ~ {:.2f}  mean={:.2f}'.format(
    combined_pm_all.min(), combined_pm_all.max(), combined_pm_all.mean()))

In [ ]:
# ── 2D 배열 변환 유틸 ─────────────────────────────────────────────────────
def to_grid_image(values, grid_df, nr, nc, fill=np.nan):
    """1D (G,) 배열 → 2D (nr, nc) 이미지 배열."""
    img = np.full((nr, nc), fill)
    for i, (r, c) in enumerate(zip(grid_df['row'], grid_df['col'])):
        img[r, c] = values[i]
    return img

# 2D 배열 생성
img_ambient  = to_grid_image(ambient_pm10_all, grid_df, NR, NC)
img_road     = to_grid_image(road_pm_all,      grid_df, NR, NC)
img_combined = to_grid_image(combined_pm_all,  grid_df, NR, NC)

print('2D 이미지 생성 완료: {} × {}'.format(NR, NC))
print('  유효 셀 비율: {:.1f}%'.format(100 * np.sum(~np.isnan(img_ambient)) / (NR*NC)))

In [ ]:
# ── 시각화 ────────────────────────────────────────────────────────────────

# 컬러맵 정의
cmap_ambient = LinearSegmentedColormap.from_list(
    'ambient', ['#FFFFFF','#EEF4FF','#BDD4FF','#6699FF','#2255CC','#001A66'])

cmap_road = LinearSegmentedColormap.from_list(
    'road', ['#FFFFFF','#FFF3CC','#FFB347','#FF6633','#CC0000','#660000'])

cmap_combined = LinearSegmentedColormap.from_list(
    'combined', ['#FFFFFF','#E8F5E9','#A5D6A7','#FFEE58','#FF9800','#D32F2F','#7B1FA2'])

# 스케일 범위
v_ambient  = (0, np.nanpercentile(img_ambient,  99))
v_road     = (0, np.nanpercentile(img_road,     99))
v_combined = (0, np.nanpercentile(img_combined, 99))

# 관측소 격자 좌표 변환 (lat/lon → 이미지 픽셀)
lat_min = grid_df['lat'].min()
lat_max = grid_df['lat'].max()
lon_min = grid_df['lon'].min()
lon_max = grid_df['lon'].max()

def latlon_to_rowcol(lat, lon, nr, nc, lat_min, lat_max, lon_min, lon_max):
    col = (lon - lon_min) / (lon_max - lon_min) * (nc - 1)
    row = (lat - lat_min) / (lat_max - lat_min) * (nr - 1)
    return row, col

sta_rows, sta_cols = latlon_to_rowcol(
    stations['위도'].values, stations['경도'].values,
    NR, NC, lat_min, lat_max, lon_min, lon_max)

# ─── Figure 생성 ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=FIG_DPI)
fig.patch.set_facecolor('#0F1923')
fig.suptitle(
    'Seoul PM10 Grid Map  |  {}'.format(actual_dt.strftime('%Y-%m-%d %H:00')),
    color='white', fontsize=14, fontweight='bold', y=1.01
)

panels = [
    (img_ambient,  cmap_ambient,  v_ambient,  'Ambient PM10 (V5 ST-GNN)',    'μg/m³'),
    (img_road,     cmap_road,     v_road,     'Road Resuspension (TwoStage)','μg/m³'),
    (img_combined, cmap_combined, v_combined, 'Combined PM10',               'μg/m³'),
]

for ax, (img, cmap, vrange, title, unit) in zip(axes, panels):
    ax.set_facecolor('#0F1923')

    # PM10 히트맵
    masked = np.ma.masked_invalid(img)
    im = ax.imshow(
        masked, origin='lower', cmap=cmap,
        vmin=vrange[0], vmax=vrange[1],
        interpolation='nearest', aspect='equal'
    )

    # 관측소 마커
    if SHOW_STATIONS:
        ax.scatter(
            sta_cols, sta_rows,
            c='white', edgecolors='#FFD700', linewidths=0.8,
            s=18, zorder=5, alpha=0.9
        )

    # 컬러바
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label(unit, color='white', fontsize=8)
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=7)

    # 통계 텍스트
    valid = img[~np.isnan(img)]
    stats_txt = 'mean={:.1f}  max={:.1f}'.format(valid.mean(), valid.max())
    ax.text(0.03, 0.97, stats_txt, transform=ax.transAxes,
            color='white', fontsize=7, va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.5))

    ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=6)
    ax.axis('off')

# 범례 (관측소)
if SHOW_STATIONS:
    legend_patch = mpatches.Patch(facecolor='white', edgecolor='#FFD700',
                                   label='PM10 Monitoring Stations (N=40)')
    fig.legend(handles=[legend_patch], loc='lower center',
               facecolor='#0F1923', edgecolor='white',
               labelcolor='white', fontsize=8, ncol=1)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=FIG_DPI, bbox_inches='tight',
            facecolor='#0F1923', pad_inches=0.1)
plt.show()
print('저장:', SAVE_PATH)

In [ ]:
# ── 시간별 변화 비교 (선택 실행) ──────────────────────────────────────────
# 같은 날 여러 시간 스냅샷 비교

COMPARE_HOURS = [6, 10, 15, 21]   # 비교할 시간대
COMPARE_DATE  = '2025-10-01'       # 비교 날짜
PANEL_TYPE    = 'combined'          # 'ambient' | 'road' | 'combined'

fig2, axes2 = plt.subplots(1, len(COMPARE_HOURS), figsize=(5*len(COMPARE_HOURS), 6), dpi=150)
fig2.patch.set_facecolor('#0F1923')
fig2.suptitle('{} — Hourly PM10 ({})'.format(COMPARE_DATE, PANEL_TYPE.upper()),
              color='white', fontsize=13, fontweight='bold')

all_vals = []
hour_imgs = []

for h in COMPARE_HOURS:
    dt_h = pd.Timestamp('{} {:02d}:00:00'.format(COMPARE_DATE, h))
    match_h = ts_lookup[ts_lookup['dt'] == dt_h]
    if len(match_h) == 0:
        diff_h = (ts_lookup['dt'] - dt_h).abs()
        match_h = ts_lookup.loc[[diff_h.idxmin()]]
    row_h = match_h.iloc[0]
    amb_h = v5_data[row_h['split']][int(row_h['local_idx'])]

    if PANEL_TYPE == 'ambient':
        vals_h = amb_h
    else:
        m_h = dt_h.month
        hh  = dt_h.hour
        wkd = dt_h.weekday()
        s_h = season_map[m_h]
        t_h = float(weather_monthly.loc[m_h, '기온'])
        hu_h = float(weather_monthly.loc[m_h, '습도'])
        id_h = int(hu_h < 40)
        mr_h = 2*np.pi*m_h/12;  hr_h = 2*np.pi*hh/24
        Xt_h = np.array([[np.sin(mr_h),np.cos(mr_h),np.sin(hr_h),np.cos(hr_h),
                           wkd,int(wkd>=5),s_h,t_h,hu_h,id_h]]*G, dtype=np.float32)
        s1_h = m1_temporal.predict(Xt_h)
        Xs_h = np.column_stack([X_spatial_base, amb_h, np.full(G, np.nan)])
        s2_h = m2_spatial.predict(imp_spatial.transform(Xs_h))
        rp_h = np.clip(transformer.inverse_transform(s1_h + s2_h), 0, None)
        vals_h = amb_h + lur_df['road_struc'].values/100.0 * rp_h if PANEL_TYPE=='combined' else rp_h

    hour_imgs.append(to_grid_image(vals_h, grid_df, NR, NC))
    all_vals.append(vals_h)

vmax_all = np.nanpercentile(np.concatenate(all_vals), 98)
cmap_sel = {'ambient':cmap_ambient,'road':cmap_road,'combined':cmap_combined}[PANEL_TYPE]

for ax2, h, img_h in zip(axes2, COMPARE_HOURS, hour_imgs):
    ax2.set_facecolor('#0F1923')
    ax2.imshow(np.ma.masked_invalid(img_h), origin='lower',
               cmap=cmap_sel, vmin=0, vmax=vmax_all,
               interpolation='nearest', aspect='equal')
    if SHOW_STATIONS:
        ax2.scatter(sta_cols, sta_rows, c='white', edgecolors='#FFD700',
                    linewidths=0.6, s=12, zorder=5, alpha=0.85)
    ax2.set_title('{:02d}:00'.format(h), color='white', fontsize=10, fontweight='bold')
    ax2.axis('off')

sm2 = ScalarMappable(cmap=cmap_sel, norm=Normalize(0, vmax_all))
sm2.set_array([])
cbar2 = fig2.colorbar(sm2, ax=axes2.tolist(), fraction=0.015, pad=0.02)
cbar2.set_label('μg/m³', color='white', fontsize=9)
cbar2.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color='white', fontsize=8)

plt.tight_layout()
hourly_path = SAVE_PATH.replace('.png', '_hourly.png')
plt.savefig(hourly_path, dpi=150, bbox_inches='tight',
            facecolor='#0F1923', pad_inches=0.1)
plt.show()
print('저장:', hourly_path)